# 03 — Phonetic Prime Hashing

**Fourth Age Paper companion notebook.** `ScalarContextPropagation` —
background for **C1/C5** (the "spelling" half of a word's address, kept
separate from the "context" half — see notebook 05).

Two versions, in the order they were actually reasoned through:

1. **The naive scheme** — the 26 letters of the English alphabet, each
   given its own prime (`a↦2, b↦3, …, z↦101`); spell a word by
   **multiplying** its letters' primes together.
2. **The shipped scheme** (`VAPMIP/wordnet_boxkite.py::spelling_code`) —
   a Gödel-positional encoding that fixes the naive scheme's one real
   flaw, demonstrated below before it is explained.

## 1. The naive scheme — a prime per letter, product of the word

`a=2, b=3, c=5, …` (the first 26 primes, one per letter), spelling =
`∏ prime(letter)` over the word.

In [1]:
import sys, os

def sieve(n):
    flags = [True] * (n + 1)
    flags[0] = flags[1] = False
    for i in range(2, int(n ** 0.5) + 1):
        if flags[i]:
            for j in range(i * i, n + 1, i):
                flags[j] = False
    return [i for i, f in enumerate(flags) if f]

_primes = sieve(200)
LETTER_PRIME = {chr(ord('a') + i): _primes[i] for i in range(26)}
print("a..z -> first 26 primes:")
print(LETTER_PRIME)

def naive_phonetic_hash(word: str) -> int:
    prod = 1
    for ch in word.lower():
        if ch.isalpha():
            prod *= LETTER_PRIME[ch]
    return prod

for w in ["cat", "act", "tac", "kite", "wind"]:
    print(f"{w!r:>8}  ->  {naive_phonetic_hash(w)}")


a..z -> first 26 primes:
{'a': 2, 'b': 3, 'c': 5, 'd': 7, 'e': 11, 'f': 13, 'g': 17, 'h': 19, 'i': 23, 'j': 29, 'k': 31, 'l': 37, 'm': 41, 'n': 43, 'o': 47, 'p': 53, 'q': 59, 'r': 61, 's': 67, 't': 71, 'u': 73, 'v': 79, 'w': 83, 'x': 89, 'y': 97, 'z': 101}
   'cat'  ->  710
   'act'  ->  710
   'tac'  ->  710
  'kite'  ->  556853
  'wind'  ->  574609


## The naive scheme's flaw — it is anagram-blind

Multiplication commutes. A product of primes carries *which* letters
appear and *how many times*, but nothing about **order** — `unique
factorisation` recovers the multiset of letters, never the sequence. Any
two words that are anagrams of each other hash identically:

In [2]:
print("naive_phonetic_hash('cat') ==", naive_phonetic_hash('cat'))
print("naive_phonetic_hash('act') ==", naive_phonetic_hash('act'))
print("equal:", naive_phonetic_hash('cat') == naive_phonetic_hash('act'))
print()
# a less trivial pair
print("naive_phonetic_hash('listen') ==", naive_phonetic_hash('listen'))
print("naive_phonetic_hash('silent') ==", naive_phonetic_hash('silent'))
print("equal:", naive_phonetic_hash('listen') == naive_phonetic_hash('silent'))
print()
print("Every anagram pair in English collides under pure multiplication.")
print("A hash that is supposed to reconstruct the SPELLING cannot lose order.")


naive_phonetic_hash('cat') == 710
naive_phonetic_hash('act') == 710
equal: True

naive_phonetic_hash('listen') == 1914801911
naive_phonetic_hash('silent') == 1914801911
equal: True

Every anagram pair in English collides under pure multiplication.
A hash that is supposed to reconstruct the SPELLING cannot lose order.


## 2. The shipped fix — Gödel positional encoding, 20 letters cycling, not 26

`VAPMIP/wordnet_boxkite.py::spelling_code` (provenance: **OURS**, marked
*"PROVISIONAL... a first working pass"* in its own docstring) fixes this
by making the **exponent** carry the letter identity and the **position**
select which prime, cycling through a 20-prime *letter tier*
(`LETTER_PRIMES`, primes `≤ 71`) rather than reserving one prime per
letter:

    position i  ->  prime  LETTER_PRIMES[i % 20]
    letter      ->  exponent  (a=1 .. z=26)
    code = ∏_i  LETTER_PRIMES[i % 20] ^ exponent(word[i])

Two different primes now anchor position, so `cat` and `act` put their
letters' exponents on **different bases** and stop colliding.

Note the tier is **20** primes (`≤ 71`), not 26 — a prime *per position*
mod 20, not one *per letter identity*. That is the one correction to
carry forward from the naive version above.

In [3]:
sys.path.insert(0, os.path.expanduser("~/Projects/ThePlace/VAPMIP"))
from wordnet_boxkite import LETTER_PRIMES, LETTER_CAP, spelling_code

print("LETTER_CAP:", LETTER_CAP, " tier size:", len(LETTER_PRIMES), " primes:", LETTER_PRIMES)
assert len(LETTER_PRIMES) == 20

for w in ["cat", "act", "listen", "silent"]:
    print(f"{w!r:>8}  spelling_code = {spelling_code(w)}")

print()
print("cat == act now?  ", spelling_code('cat') == spelling_code('act'))
print("listen == silent? ", spelling_code('listen') == spelling_code('silent'))


LETTER_CAP: 71  tier size: 20  primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]
   'cat'  spelling_code = 2288818359375000
   'act'  spelling_code = 5149841308593750
'listen'  spelling_code = 77805891137496681806187293144874121384823203125000000000000
'silent'  spelling_code = 305601935552050069509298342748495166804755637888000000000000

cat == act now?   False
listen == silent?  False


## Round trip — recover the spelling from the code

Position `i` uses `LETTER_PRIMES[i % 20]`; for words of at most 20 letters
every position's prime is distinct, so factoring the code recovers each
letter's exponent (and hence the letter) unambiguously, in order.

In [4]:
def spell_decode(code: int, length: int) -> str:
    letters = []
    for i in range(length):
        p = LETTER_PRIMES[i % len(LETTER_PRIMES)]
        e = 0
        while code % p == 0:
            code //= p
            e += 1
        letters.append(chr(ord('a') + e - 1) if 1 <= e <= 26 else '?')
    return ''.join(letters)

for w in ["cat", "act", "kite", "windspeed", "reconstructible"]:
    code = spelling_code(w)
    dec = spell_decode(code, len(w))
    print(f"{w!r:>18}  ->  decoded {dec!r:>18}  match: {dec == w}")


             'cat'  ->  decoded              'cat'  match: True
             'act'  ->  decoded              'act'  match: True
            'kite'  ->  decoded             'kite'  match: True
       'windspeed'  ->  decoded        'windspeed'  match: True
 'reconstructible'  ->  decoded  'reconstructible'  match: True


## Measured on the live vocabulary (30k sample)

`spelling_code` produces genuinely large integers (one prime-power factor
per letter position, exponent up to 26) — bignum arithmetic over the full
347,119-word store is expensive enough that the original measurement in
`bench/roundtrip_results.txt` samples **30,000** words rather than the
full store. Reproduced here at the same scale, for the same reason:

In [5]:
import struct, mmap, random

STORE = os.path.expanduser("~/Projects/ThePlace/VAPMIP/PtolC/monad3_c.bin")

def iter_words(path):
    f = open(path, "rb")
    mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
    hdr = struct.Struct("<8s6I16d13Q")
    vals = hdr.unpack_from(mm, 0)
    n_words = vals[2]
    off = vals[23:]
    o_blob, o_rec = off[0], off[1]
    rec = struct.Struct("<iiii")
    def name(o):
        e = mm.find(b"\x00", o_blob + o)
        return mm[o_blob + o:e].decode("utf-8", "replace")
    for i in range(n_words):
        noff, ei, wi, pi = rec.unpack_from(mm, o_rec + i * 16)
        yield name(noff)

random.seed(20260831)
all_words = list(iter_words(STORE))
sample = random.sample(all_words, 30_000)

exact = le20 = exact_le20 = gt20 = total = 0
for w in sample:
    alpha = "".join(ch for ch in w.lower() if ch.isalpha())
    if not alpha:
        continue
    total += 1
    dec = spell_decode(spelling_code(w), len(alpha))
    ok = (dec == alpha)
    exact += ok
    if len(alpha) <= 20:
        le20 += 1
        exact_le20 += ok
    else:
        gt20 += 1

print(f"exact word recovery: {exact:,}/{total:,}  ({exact/total*100:.3f}%)")
print(f"words <=20 alpha chars: {le20:,}  exact {exact_le20:,} ({exact_le20/le20*100:.3f}%)")
print(f"words  >20 alpha chars: {gt20:,}  (position cycle wraps at 20 -> exponents "
      f"add on the reused prime -> lossy, by construction)")


exact word recovery: 27,375/28,728  (95.290%)
words <=20 alpha chars: 28,296  exact 27,375 (96.745%)
words  >20 alpha chars: 432  (position cycle wraps at 20 -> exponents add on the reused prime -> lossy, by construction)


## Summary

| scheme | order-sensitive | round trip | note |
|---|---|---|---|
| naive: 26 primes, one per letter, product | **no** — anagram-blind | fails on any anagram pair | illustrates *why* position must be encoded |
| shipped: `spelling_code`, 20-prime tier, Gödel positional | yes | exact for ≤20-char words (measured above); lossy beyond via prime-cycle wraparound | **OURS**, marked provisional in-source |

The correction to carry into the rest of the series: the letter tier is
**20 primes (≤71)**, cycling by position, not 26 primes assigned one per
letter — the 26 only shows up as the **exponent range** (`a=1..z=26`),
not the prime count.